# 09_model_explainability

This notebook provides board-ready explanations for models used in the IBRD World Loans project. It saves figures to `reports/figures/` and writes a markdown report to `reports/model_explainability_report.md`.

In [2]:
# Setup imports and paths
import pathlib, pickle, joblib, os
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, plotly.express as px
from sklearn.preprocessing import RobustScaler
import shap

DATA_PATH = pathlib.Path('data/processed/ibrd_clean.csv')
MODELS_DIR = pathlib.Path('models_pickle')
FIGURES_DIR = pathlib.Path('reports/figures')
REPORT_MD = pathlib.Path('reports/model_explainability_report.md')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
(REPORT_MD.parent).mkdir(parents=True, exist_ok=True)

In [5]:
# 1) Load models
models = {}
model_files = {
    'repayment_xgboost': MODELS_DIR / 'repayment_xgboost.pkl',
    'country_kmeans': MODELS_DIR / 'country_kmeans.pkl',
    'anomaly_isolation_forest': MODELS_DIR / 'anomaly_isolation_forest.pkl',
}
for name, path in model_files.items():
    if path.exists():
        try:
            models[name] = joblib.load(path)
        except Exception:
            with open(path, 'rb') as f:
                models[name] = pickle.load(f)
    else:
        print(f'Warning: model file not found: {path}')

models.keys()

dict_keys([])

In [14]:
# Load cleaned data
if DATA_PATH.exists():
    df = pd.read_csv(DATA_PATH, low_memory=False)
else:
    # Try common repo-relative locations when the notebook is launched from a subfolder
    candidates = [
        DATA_PATH,
        pathlib.Path.cwd() / DATA_PATH,
        pathlib.Path.cwd().parent / DATA_PATH,
        pathlib.Path("/home/rigii/ATA") / DATA_PATH,
    ]
    for base in [pathlib.Path.cwd(), *list(pathlib.Path.cwd().parents)]:
        candidates.append(base / "data" / "processed" / "ibrd_clean.csv")

    resolved = next((p for p in candidates if p.exists()), None)
    if resolved is None:
        raise FileNotFoundError(
            f"Missing data file: {DATA_PATH}. Tried: {[str(p) for p in candidates]}"
        )

    df = pd.read_csv(resolved, low_memory=False)

df.shape

(9518, 54)

## Global Feature Importance (XGBoost + SHAP)

In [15]:
xgb = models.get('repayment_xgboost')
if xgb is None:
    print('repayment_xgboost not loaded; skip global importance')
else:
    if hasattr(xgb, 'feature_names_in_'):
        feature_names = list(xgb.feature_names_in_)
    else:
        drop_cols = ['Loan Number','Country / Economy','Approval Date','Approval Year']
        feature_names = [c for c in df.select_dtypes(include=[np.number]).columns if c not in drop_cols]
    try:
        importances = xgb.feature_importances_
        fi = pd.Series(importances, index=feature_names).sort_values(ascending=False)
        fig,ax = plt.subplots(figsize=(8,6)); fi.head(30).plot.bar(ax=ax); plt.tight_layout(); fig.savefig(FIGURES_DIR/'global_feature_importance.png', dpi=200); print('Saved global_feature_importance.png')
    except Exception as e:
        print('Could not plot built-in importances:', e)
    try:
        X = df[feature_names].fillna(0)
        expl = shap.TreeExplainer(xgb)
        shap_vals = expl.shap_values(X)
        plt.figure(figsize=(8,6))
        shap.summary_plot(shap_vals, X, show=False)
        plt.tight_layout(); plt.savefig(FIGURES_DIR/'global_shap_summary.png', dpi=200); print('Saved global_shap_summary.png')
    except Exception as e:
        print('SHAP summary failed:', e)

repayment_xgboost not loaded; skip global importance


## Local Explanations — SHAP Waterfall

In [16]:
if 'feature_names' not in globals():
    drop_cols = ['Loan Number','Country / Economy','Approval Date','Approval Year']
    feature_names = [c for c in df.select_dtypes(include=[np.number]).columns if c not in drop_cols]
X = df[feature_names].fillna(0)
indices = []
if xgb is not None:
    try:
        probs = xgb.predict_proba(X)[:,1]
        idx_repay = int(np.argmax(probs)); idx_risk = int(np.argmin(probs))
    except Exception:
        probs = xgb.predict(X); idx_repay, idx_risk = 0,1
else:
    idx_repay, idx_risk = 0,1
iso = models.get('anomaly_isolation_forest')
an_idx = None
if iso is not None:
    try:
        an_candidates = np.where(iso.predict(X)==-1)[0]
        an_idx = int(an_candidates[0]) if len(an_candidates) else None
    except Exception:
        an_idx = None
indices = [i for i in [idx_repay, idx_risk, an_idx] if i is not None][:3]
for i, ridx in enumerate(indices, start=1):
    try:
        expl = shap.Explainer(xgb)
        svals = expl(X.iloc[[ridx]])
        plt.figure(figsize=(6,4))
        shap.plots.waterfall(svals[0], show=False)
        plt.tight_layout(); plt.savefig(FIGURES_DIR/f'local_explanation_loan_{i}.png', dpi=200)
        print('Saved local_explanation_loan_', i)
    except Exception as e:
        print('Local SHAP failed for', ridx, e)

Local SHAP failed for 0 The passed model is not callable and cannot be analyzed directly with the given masker! Model: None
Local SHAP failed for 1 The passed model is not callable and cannot be analyzed directly with the given masker! Model: None


## Cluster Interpretation (4 clusters)

In [17]:
# Fallback: create 4 clusters by repayment_ratio quartiles and save summary
country_ag = df.groupby('Country / Economy').agg(commitments=('Original Principal Amount (US$)','sum'), repayments=('Repaid to IBRD (US$)','sum'), loans_count=('Loan Number','count')).reset_index()
country_ag['repayment_ratio'] = country_ag['repayments'] / country_ag['commitments'].replace(0, np.nan)
country_ag['repayment_ratio'] = country_ag['repayment_ratio'].fillna(0)
country_ag['cluster'] = pd.qcut(country_ag['repayment_ratio'].rank(method='first'), 4, labels=False)
out = FIGURES_DIR/'cluster_summary.csv'
rows = []
for cl in sorted(country_ag['cluster'].unique()):
    sub = country_ag[country_ag['cluster']==cl]
    rows.append({'cluster':int(cl),'n_countries':len(sub),'avg_commitments':float(sub['commitments'].mean()),'avg_repayment_ratio':float(sub['repayment_ratio'].mean()),'examples':'; '.join(sub['Country / Economy'].head(5).tolist())})
pd.DataFrame(rows).to_csv(out, index=False)
print('Saved', out)

Saved reports/figures/cluster_summary.csv


## Anomaly Model Explanation

In [18]:
iso = models.get('anomaly_isolation_forest')
if iso is None:
    print('No anomaly model loaded; skipping')
else:
    try:
        scores = iso.decision_function(X)
        top5 = np.argsort(scores)[:5]
        print('Top5 anomaly rows:', top5)
    except Exception as e:
        print('Anomaly explanation failed:', e)

No anomaly model loaded; skipping


## Model Cards & Business Recommendations
The notebook writes a combined markdown report `reports/model_explainability_report.md` with model cards and 10 business recommendations.

In [19]:
rows = len(df)
date_range = (df['Approval Date'].min() if 'Approval Date' in df.columns else None, df['Approval Date'].max() if 'Approval Date' in df.columns else None)

def card_for(name):
    return f'### {name}\n- Purpose: TBD\n- Training rows: {rows}\n- Date range: {date_range}\n- Features: TBD\n- Metrics: TBD\n- Limitations: TBD\n- Ethical considerations: TBD\n- Intended use: TBD\n'

md_parts = ['# Model Explainability Report\n', f'Data rows: {rows}\n', f'Approval Date range: {date_range}\n']
for n in models.keys():
    md_parts.append(card_for(n))
md_parts.append('## Business Recommendations\n')
md_parts.extend([f'- Recommendation {i+1}\n' for i in range(10)])
with open(REPORT_MD, 'w', encoding='utf8') as f:
    f.write('\n'.join(md_parts))
print('Wrote', REPORT_MD)

Wrote reports/model_explainability_report.md
